# Generative AI Tools, Techniques & Applications – Practical Notebook

**ADPCC704 | AISSMSIOIT**

This notebook contains short, beginner-friendly code for all 12 practicals from the supplied lab manual. Run each practical separately. Internet access is needed for pretrained model downloads.

**Recommended:** Use Google Colab for Practicals 5–9. Select **Runtime → Change runtime type → T4 GPU** for image-generation practicals when available.

## Setup – Install required libraries

Run this cell once. Some packages may already be installed in Google Colab.

In [ ]:
%pip install -q tensorflow pandas numpy matplotlib seaborn scikit-learn gensim faiss-cpu sentence-transformers torch transformers datasets peft accelerate diffusers rouge-score

## Practical 1 – TensorFlow 2: Tensor creation, manipulation and eager execution

**Objective:** Create tensors and perform arithmetic and reshape operations.

In [ ]:
import tensorflow as tf

a = tf.constant([10, 20, 30])
b = tf.constant([1, 2, 3])

print("Addition:", tf.add(a, b).numpy())
print("Subtraction:", tf.subtract(a, b).numpy())
print("Multiplication:", tf.multiply(a, b).numpy())

matrix = tf.constant([[1, 2], [3, 4]])
print("Reshaped:", tf.reshape(matrix, [4]).numpy())
print("Eager execution:", tf.executing_eagerly())

**Expected output**
```text
Addition: [11 22 33]
Subtraction: [ 9 18 27]
Multiplication: [10 40 90]
Reshaped: [1 2 3 4]
Eager execution: True
```

## Practical 2 – Data preprocessing using Pandas and NumPy

**Objective:** Fill missing values, normalize numerical columns and one-hot encode categories.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

data = {
    "Name": ["A", "B", "C", "D"],
    "Age": [20, 25, None, 35],
    "Marks": [60, 70, 80, 90],
    "City": ["Pune", "Mumbai", "Pune", "Delhi"]
}
df = pd.DataFrame(data)

df["Age"] = df["Age"].fillna(df["Age"].mean())

scaler = MinMaxScaler()
df[["Age", "Marks"]] = scaler.fit_transform(df[["Age", "Marks"]])

df = pd.get_dummies(df, columns=["City"], dtype=int)
print(df.round(3).to_string(index=False))

**Expected result:** Missing Age is replaced with its mean (26.667), Age and Marks are scaled to the range 0–1, and City becomes three binary columns.

## Practical 3 – Data visualization using Matplotlib and Seaborn

**Objective:** Plot a histogram, scatter plot and correlation heatmap.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.DataFrame({
    "Age": [20, 22, 25, 28, 30],
    "Marks": [60, 65, 70, 80, 90],
    "Hours": [2, 3, 4, 5, 6]
})

plt.figure(figsize=(5, 3))
plt.hist(df["Marks"], bins=5, edgecolor="black")
plt.title("Marks Distribution")
plt.xlabel("Marks")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(5, 3))
plt.scatter(df["Hours"], df["Marks"])
plt.title("Study Hours vs Marks")
plt.xlabel("Study Hours")
plt.ylabel("Marks")
plt.grid()
plt.show()

plt.figure(figsize=(5, 3))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

**Expected output:** Three plots: a marks histogram, a positive-trend scatter plot, and a correlation heatmap.

## Practical 4 – Word embeddings and similarity search using FAISS

**Objective:** Train Word2Vec embeddings and search for nearby word vectors.

In [ ]:
from gensim.models import Word2Vec
import faiss
import numpy as np

sentences = [
    ["king", "queen", "man", "woman"],
    ["dog", "cat", "animal", "pet"],
    ["apple", "mango", "fruit", "food"],
    ["king", "man", "royal"],
    ["queen", "woman", "royal"]
]

model = Word2Vec(
    sentences, vector_size=50, window=3,
    min_count=1, workers=1, seed=42, epochs=100
)

words = list(model.wv.index_to_key)
vectors = np.array([model.wv[w] for w in words], dtype="float32")

index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

query = model.wv["king"].reshape(1, -1)
distances, ids = index.search(query, 3)

print("Nearest words to 'king':")
for i in ids[0]:
    print(words[i])

**Expected result:** The nearest result is `king` itself; other nearby words depend on the small corpus and training.

## Practical 5 – Image generation using a pretrained DDPM model

**Objective:** Generate MNIST-like handwritten digits with a pretrained diffusion model. This example uses Hugging Face Diffusers (PyTorch) for pretrained DDPM inference.

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import DDPMPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = DDPMPipeline.from_pretrained("google/ddpm-ema-mnist-28")
pipe = pipe.to(device)

images = pipe(batch_size=4).images

fig, axes = plt.subplots(1, 4, figsize=(8, 2))
for ax, image in zip(axes, images):
    ax.imshow(image, cmap="gray")
    ax.axis("off")
plt.suptitle("Generated MNIST Digits")
plt.show()

**Expected output:** Four synthetic grayscale handwritten-digit images. The model download requires internet access; generation may be slow on CPU.

## Practical 6 – Text-to-image generation using a pretrained diffusion model

**Objective:** Generate an image from a text prompt. Stable Diffusion Turbo uses text conditioning; its text encoder is CLIP-based, but this is not the original gradient-based CLIP-guided diffusion algorithm.

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import AutoPipelineForText2Image

if not torch.cuda.is_available():
    raise RuntimeError("For this practical, select a GPU runtime in Google Colab.")

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16
).to("cuda")

prompt = "A beautiful futuristic city with flying cars"
image = pipe(
    prompt,
    num_inference_steps=4,
    guidance_scale=0.0
).images[0]

plt.figure(figsize=(7, 5))
plt.imshow(image)
plt.axis("off")
plt.title("Text-to-Image Generation")
plt.show()

**Expected output:** An AI-generated image resembling the city described in the prompt. Model access and GPU memory are required.

## Practical 7 – Fine-tuning DistilGPT-2 using LoRA

**Objective:** Apply LoRA to a small pretrained causal language model, train briefly and generate text. Training is intended for a small demonstration dataset.

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

model_name = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["c_attn"],
    fan_in_fan_out=True
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

texts = [
    "Artificial intelligence enables machines to perform intelligent tasks.",
    "Generative AI creates new text, images and other content.",
    "Machine learning finds patterns in data.",
    "Deep learning uses multi-layer neural networks."
]
dataset = Dataset.from_dict({"text": texts})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

args = TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    report_to="none",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False
    )
)
trainer.train()

inputs = tokenizer("Artificial intelligence", return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
output = model.generate(
    **inputs, max_new_tokens=30,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

**Expected output:** Training logs followed by generated text. The exact completion varies; a tiny dataset is only for demonstrating the LoRA workflow.

## Practical 8 – Prompt engineering for summarization and question answering

**Objective:** Use pretrained transformer pipelines and evaluate a summary with ROUGE-1.

In [ ]:
from transformers import pipeline
from rouge_score import rouge_scorer

text = (
    "Artificial Intelligence is a branch of computer science. "
    "It enables machines to perform tasks such as learning, "
    "reasoning and problem solving."
)

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)
summary = summarizer(
    text, max_length=40, min_length=8, do_sample=False
)[0]["summary_text"]
print("Summary:", summary)

qa = pipeline(
    "question-answering",
    model="distilbert/distilbert-base-cased-distilled-squad"
)
answer = qa(
    question="What is Artificial Intelligence?",
    context=text
)["answer"]
print("Answer:", answer)

reference = "Artificial Intelligence enables machines to learn and reason."
scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
score = scorer.score(reference, summary)
print("ROUGE-1 F1:", round(score["rouge1"].fmeasure, 3))

**Expected output:** A short summary, an answer to the question, and a numeric ROUGE-1 F1 score. Text and score can vary by model version.

## Practical 9 – Retrieval-Augmented Generation (RAG) using FAISS

**Objective:** Retrieve the most relevant document using embeddings and use it as context for an LLM.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

documents = [
    "Artificial Intelligence enables machines to perform intelligent tasks.",
    "Machine Learning is a subset of Artificial Intelligence.",
    "Deep Learning uses multi-layer neural networks.",
    "Python is widely used for AI development."
]

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.encode(documents, convert_to_numpy=True).astype("float32")

index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

question = "What is Machine Learning?"
query_vector = embedder.encode([question]).astype("float32")
distances, ids = index.search(query_vector, 1)
context = documents[ids[0][0]]
print("Retrieved document:", context)

generator = pipeline("text2text-generation", model="google/flan-t5-small")
prompt = f"Answer the question using this context. Context: {context} Question: {question}"
answer = generator(prompt, max_new_tokens=50)[0]["generated_text"]
print("Generated answer:", answer)

**Expected output:** The retrieved sentence should identify Machine Learning as a subset of AI, followed by a short generated answer. This is a minimal RAG demonstration.

## Practical 10 – Model optimization using pruning and quantization

**Objective:** Apply weight pruning and dynamic quantization to a pretrained model and compare inference time. This example uses a small DistilBERT sentiment classifier.

In [ ]:
import copy
import time
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.utils import prune
from torch.ao.quantization import quantize_dynamic

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).eval()

inputs = tokenizer("Generative AI is useful.", return_tensors="pt")

def measure(model, inputs):
    start = time.perf_counter()
    with torch.no_grad():
        model(**inputs)
    return time.perf_counter() - start

original_time = measure(model, inputs)

pruned_model = copy.deepcopy(model)
for module in pruned_model.modules():
    if isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=0.3)
        prune.remove(module, "weight")

quantized_model = quantize_dynamic(
    pruned_model, {torch.nn.Linear}, dtype=torch.qint8
).eval()

optimized_time = measure(quantized_model, inputs)

print("Original inference time:", round(original_time, 4), "seconds")
print("Pruned + quantized inference time:", round(optimized_time, 4), "seconds")
print("Optimization completed.")

**Expected output:** Two measured inference times and a completion message. Times vary by hardware; optimization does not guarantee a speed-up for every machine.

## Practical 11 – Bias detection and mitigation using output filtering

**Objective:** Generate text, check for selected stereotype-related phrases and flag potentially biased output. This simple keyword filter is only a classroom demonstration, not a complete fairness test.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

prompt = "A nurse is"
generated = generator(
    prompt,
    max_new_tokens=25,
    do_sample=False,
    pad_token_id=generator.tokenizer.eos_token_id
)[0]["generated_text"]

print("Original output:", generated)

flagged_phrases = ["only women", "only men", "inferior"]

def filter_bias(text):
    lower_text = text.lower()
    if any(phrase in lower_text for phrase in flagged_phrases):
        return "Output flagged for bias review."
    return text

print("Filtered output:", filter_bias(generated))

**Expected output:** The original generated text and either the same text or a bias-review warning. The result depends on the model's generated text.

## Practical 12 – Autonomous AI agent and multi-agent interaction

**Objective:** Simulate multiple simple agents that collect or share resources and compare their final scores.

In [ ]:
import random

random.seed(7)

agents = {
    "Agent 1": 0,
    "Agent 2": 0,
    "Agent 3": 0
}

for round_no in range(1, 6):
    print("\nRound:", round_no)

    for name in agents:
        action = random.choice(["Collect", "Share"])

        if action == "Collect":
            agents[name] += 1
            print(name, "collected a resource")
        else:
            print(name, "shared resources")

print("\nFinal scores:")
for name, score in agents.items():
    print(name, ":", score)

**Expected output:** Five rounds of agent actions followed by each agent's final resource score. The random seed makes the example repeatable.

## Practical completion checklist

- [ ] 1. TensorFlow tensors and eager execution
- [ ] 2. Data cleaning, normalization and encoding
- [ ] 3. Histogram, scatter plot and heatmap
- [ ] 4. Word2Vec and FAISS similarity search
- [ ] 5. Pretrained DDPM image generation
- [ ] 6. Text-to-image diffusion
- [ ] 7. DistilGPT-2 LoRA fine-tuning
- [ ] 8. Prompting, summarization, QA and ROUGE
- [ ] 9. FAISS-based RAG
- [ ] 10. Pruning and quantization
- [ ] 11. Bias detection and output filtering
- [ ] 12. Autonomous multi-agent simulation

**Tip:** Run the setup cell first, then execute each practical from top to bottom. Pretrained models may take time to download on the first run.